# Подробный анализ одного ROS 2 bag

Notebook декодирует выбранный прогон и показывает:

- положение ручки контроллера;
- скорости передней и задней тележек;
- GNSS master/rover как отдельный ground truth;
- временные шкалы и смещения `recorded_ns − header_ns`;
- локальную GNSS-траекторию в метрах;
- синхронизированное сравнение колесной и GNSS-скорости.

> Колёсные скорости и контроллер — разрешённые входы оценивателя. GNSS используется здесь исключительно для офлайн-аналитики и метрик.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from bag_analysis import (
    build_speed_comparison,
    load_bag_frames,
    read_bag_metadata,
    speed_error_metrics,
    stream_timing_summary,
)

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")

## 1. Параметры

Измените `BAG_ID` и перезапустите все ячейки. Для первой проверки подходит короткий динамический прогон `30618_0d865417`.

In [ ]:
BAG_ID = "30618_0d865417"
SYNC_TOLERANCE_MS = 150.0
WHEEL_DISAGREEMENT_THRESHOLD_MPS = 2.0

In [ ]:
frames = load_bag_frames(BAG_ID)
metadata = pd.Series(read_bag_metadata(frames.bag_path))

print(f"Bag: {frames.bag_id}")
print(f"Поддерживаемых сообщений: {len(frames.records):,}")
display(metadata.to_frame("value"))

## 2. Структура нормализованных данных

In [ ]:
table_sizes = pd.Series({
    "control": len(frames.control),
    "wheels": len(frames.wheels),
    "gnss_fix": len(frames.gnss_fix),
    "gnss_velocity": len(frames.gnss_velocity),
})

display(table_sizes.to_frame("rows"))
display(frames.records.head(10))

## 3. Время и качество потока

`header_ns` — время измерения из сообщения. `recorded_ns` — время записи сообщения в bag. Разность между ними может включать разные часы, стартовый backlog и транспортные задержки, поэтому это диагностический показатель, а не готовая оценка latency.

In [ ]:
timing = stream_timing_summary(frames)
timing

In [ ]:
delay_plot = frames.records[["topic", "delay_ms"]].dropna().copy()
low, high = delay_plot["delay_ms"].quantile([0.01, 0.99])
delay_plot["delay_ms_clipped"] = delay_plot["delay_ms"].clip(low, high)

plt.figure(figsize=(16, 6))
sns.boxplot(delay_plot, x="delay_ms_clipped", y="topic", showfliers=False)
plt.title("recorded_ns − header_ns (обрезаны 1-й и 99-й процентили для масштаба)")
plt.xlabel("Разность времени, мс")
plt.ylabel("")
plt.show()

## 4. Все основные сигналы во времени

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(17, 12), sharex=True)

for wheel_id, part in frames.wheels.groupby("wheel_id"):
    axes[0].plot(part["time_s"], part["velocity_mps"], label=wheel_id, linewidth=1.2)
axes[0].set_ylabel("Скорость тележки, м/с")
axes[0].legend()

if not frames.gnss_velocity.empty:
    for receiver, part in frames.gnss_velocity.groupby("receiver"):
        axes[1].plot(
            part["time_s"],
            part["horizontal_speed_mps"],
            label=f"GNSS {receiver}",
            linewidth=1.1,
        )
    axes[1].legend()
else:
    axes[1].text(0.5, 0.5, "В этом bag нет GNSS velocity", ha="center", va="center", transform=axes[1].transAxes)
axes[1].set_ylabel("GNSS speed, м/с")

axes[2].step(
    frames.control["time_s"],
    frames.control["position"],
    where="post",
    label="Положение контроллера",
)
axes[2].axhline(0, color="black", linewidth=0.8)
axes[2].set(ylabel="Позиция −15…15", xlabel="Время по header от начала, с")
axes[2].legend()

fig.suptitle(f"Сигналы {BAG_ID}", y=1.01)
fig.tight_layout()
plt.show()

## 5. Согласованность тележек

In [ ]:
wheel_pair = (
    frames.wheels.pivot_table(
        index="header_ns",
        columns="wheel_id",
        values="velocity_mps",
        aggfunc="last",
    )
    .sort_index()
)
wheel_pair["time_s"] = (wheel_pair.index - frames.origin_ns) / 1e9

if {"front_bogie", "rear_bogie"}.issubset(wheel_pair.columns):
    wheel_pair["difference_mps"] = wheel_pair["front_bogie"] - wheel_pair["rear_bogie"]
    wheel_pair["abs_difference_mps"] = wheel_pair["difference_mps"].abs()
    display(wheel_pair[["front_bogie", "rear_bogie", "difference_mps"]].describe())

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    axes[0].plot(wheel_pair["time_s"], wheel_pair["difference_mps"])
    axes[0].axhline(0, color="black", linewidth=0.8)
    axes[0].set(title="Передняя − задняя тележка", xlabel="Время, с", ylabel="Разность, м/с")
    sns.histplot(wheel_pair["difference_mps"].dropna(), bins=30, ax=axes[1])
    axes[1].set(title="Распределение разности", xlabel="Разность, м/с")
    fig.tight_layout()
    plt.show()
else:
    print("Нет синхронной пары front/rear для сравнения.")

## 6. GNSS-положение в локальной системе

Координаты преобразованы из WGS84 в локальную азимутальную проекцию относительно первого валидного `master` fix (или первого доступного fix). Это удобно для EDA, но перед использованием в метриках необходимо согласовать систему координат с контрактом жюри. `master` и `rover` — разные приёмники/точки, их нельзя автоматически считать одной позицией кузова.

In [ ]:
if frames.gnss_fix.empty or frames.gnss_fix["local_x_m"].notna().sum() == 0:
    print("В этом bag нет валидных GNSS fix.")
else:
    print("Начало локальной системы:", frames.gnss_fix.attrs.get("reference"))
    display(
        frames.gnss_fix.groupby(["receiver", "status", "position_covariance_type"], dropna=False)
        .size()
        .rename("samples")
        .to_frame()
    )

    fig, axes = plt.subplots(1, 2, figsize=(16, 7))
    for receiver, part in frames.gnss_fix.dropna(subset=["local_x_m", "local_y_m"]).groupby("receiver"):
        axes[0].plot(part["local_x_m"], part["local_y_m"], label=receiver)
        axes[0].scatter(part.iloc[0]["local_x_m"], part.iloc[0]["local_y_m"], marker="o", s=50)
        axes[0].scatter(part.iloc[-1]["local_x_m"], part.iloc[-1]["local_y_m"], marker="x", s=70)
        axes[1].plot(part["time_s"], part["local_z_m"], label=receiver)
    axes[0].set(title="Локальная траектория (начало ○, конец ×)", xlabel="X, м", ylabel="Y, м")
    axes[0].axis("equal")
    axes[0].legend()
    axes[1].set(title="Относительная высота", xlabel="Время, с", ylabel="Z, м")
    axes[1].legend()
    fig.tight_layout()
    plt.show()

## 7. Колёсная скорость против GNSS truth

GNSS даёт модуль горизонтальной скорости. Поэтому ниже с ним сравнивается **модуль медианы тележек**. Это описательная проверка датчиков, а не метрика готовой модели: знак движения теряется, а временная синхронизация выполняется ближайшим соседом в пределах `SYNC_TOLERANCE_MS`.

In [ ]:
comparison = build_speed_comparison(frames, tolerance_ms=SYNC_TOLERANCE_MS)
metrics = speed_error_metrics(comparison)

if comparison.empty or metrics.empty:
    print("Недостаточно колесных или GNSS velocity данных для сравнения.")
else:
    display(metrics.to_frame("value"))

    fig, axes = plt.subplots(1, 3, figsize=(19, 5))
    axes[0].plot(comparison["time_s"], comparison["wheel_speed_magnitude_mps"], label="|median wheels|")
    axes[0].plot(comparison["time_s"], comparison["gnss_speed_mps"], label="median GNSS", alpha=0.85)
    axes[0].set(title="Синхронизированные скорости", xlabel="Время, с", ylabel="м/с")
    axes[0].legend()

    axes[1].plot(comparison["time_s"], comparison["speed_error_mps"])
    axes[1].axhline(0, color="black", linewidth=0.8)
    axes[1].set(title="Ошибка wheels − GNSS", xlabel="Время, с", ylabel="м/с")

    paired = comparison.dropna(subset=["wheel_speed_magnitude_mps", "gnss_speed_mps"])
    axes[2].scatter(paired["gnss_speed_mps"], paired["wheel_speed_magnitude_mps"], s=14, alpha=0.65)
    limit = max(paired["gnss_speed_mps"].max(), paired["wheel_speed_magnitude_mps"].max())
    axes[2].plot([0, limit], [0, limit], "k--", linewidth=1)
    axes[2].set(title="Wheels vs GNSS", xlabel="GNSS, м/с", ylabel="Колёса, м/с")

    fig.tight_layout()
    plt.show()

## 8. Связь контроллера с динамикой GNSS

In [ ]:
if frames.gnss_velocity.empty or frames.control.empty:
    print("Нет данных для анализа контроллера и GNSS.")
else:
    truth = (
        frames.gnss_velocity.pivot_table(
            index="header_ns",
            columns="receiver",
            values="horizontal_speed_mps",
            aggfunc="last",
        )
        .sort_index()
    )
    truth["speed_mps"] = truth.median(axis=1, skipna=True)
    truth = truth.reset_index()
    truth["dt_s"] = truth["header_ns"].diff() / 1e9
    truth["acceleration_mps2"] = truth["speed_mps"].diff() / truth["dt_s"]
    truth.loc[~truth["dt_s"].between(0.02, 1.0), "acceleration_mps2"] = np.nan

    dynamics = pd.merge_asof(
        truth.sort_values("header_ns"),
        frames.control[["header_ns", "position"]].sort_values("header_ns"),
        on="header_ns",
        direction="backward",
        tolerance=int(0.5e9),
    )

    controller_summary = dynamics.groupby("position").agg(
        samples=("speed_mps", "count"),
        median_speed_mps=("speed_mps", "median"),
        median_acceleration_mps2=("acceleration_mps2", "median"),
        acceleration_p10=("acceleration_mps2", lambda values: values.quantile(0.10)),
        acceleration_p90=("acceleration_mps2", lambda values: values.quantile(0.90)),
    )
    display(controller_summary)

    plt.figure(figsize=(16, 6))
    sns.boxplot(
        dynamics.dropna(subset=["position", "acceleration_mps2"]),
        x="position",
        y="acceleration_mps2",
        showfliers=False,
    )
    plt.title("Распределение GNSS-ускорения по позиции контроллера")
    plt.xlabel("Позиция контроллера")
    plt.ylabel("Ускорение, м/с²")
    plt.show()

## 9. Автоматические флаги для дальнейшего исследования

In [ ]:
flags = {
    "negative_wheel_samples": int((frames.wheels["velocity_mps"] < 0).sum()),
    "non_finite_wheel_samples": int((~frames.wheels["velocity_finite"].eq(True)).sum()),
    "invalid_gnss_fixes": int((~frames.gnss_fix["has_fix"].eq(True)).sum()) if not frames.gnss_fix.empty else 0,
    "header_reversals": int(timing["header_reversal_count"].sum()),
    "duplicate_header_timestamps": int(timing["duplicate_header_count"].sum()),
}

if "abs_difference_mps" in wheel_pair:
    flags["large_wheel_disagreement_samples"] = int(
        (wheel_pair["abs_difference_mps"] > WHEEL_DISAGREEMENT_THRESHOLD_MPS).sum()
    )
else:
    flags["large_wheel_disagreement_samples"] = 0

pd.Series(flags).to_frame("count")

## Интерпретация

- Большая разность тележек может означать буксование/юз, но также требует проверки времени и качества обоих каналов.
- GNSS covariance с типом `UNKNOWN` и нулевыми значениями нельзя трактовать как идеальную точность.
- GNSS horizontal speed не содержит знак движения; для знаковой продольной скорости нужна проекция вектора на направление пути.
- Для выводов о модели анализируйте несколько bag и разделяйте train/validation/test целыми сессиями.
- Все численные пороги в notebook — исследовательские параметры, а не готовые safety-пороги production-системы.